In [ ]:
import os
import pandas as pd

In [ ]:
area = "extremadura"
scenario_rf = "C6"

# C2 e C4: area = "centro"
# C6: area = "centro" ou "extremadura"


# C2 — LR do C1 | ICNF | 1995–2024
if scenario_rf == "C2":
    scenario_lr = "C1"
    train_period = "1995_2024"


# C4 — LR do C3 | ICNF | 2008–2024
elif scenario_rf == "C4":
    scenario_lr = "C3"
    train_period = "2008_2024"


# C6 — LR do C5 | EFFIS | 2008–2024
elif scenario_rf == "C6":
    scenario_lr = "C5"
    train_period = "2008_2024"


rf_folder = (
    f"/code/data/results/{area}/"
    f"{scenario_rf}/lri_model"
)

lr_final_folder = (
    f"/code/data/results/{area}/"
    f"{scenario_lr}/final"
)

train_folder = os.path.join(
    rf_folder,
    f"train_f{train_period}"
)

xlsx = os.path.join(
    rf_folder,
    "models.xlsx"
)


os.makedirs(rf_folder, exist_ok=True)
os.makedirs(
    os.path.join(rf_folder, "npy"),
    exist_ok=True
)
os.makedirs(
    os.path.join(rf_folder, "models"),
    exist_ok=True
)
os.makedirs(
    os.path.join(rf_folder, "class"),
    exist_ok=True
)


print("Cenário RF:", scenario_rf)
print("Área:", area)
print("Cenário LR:", scenario_lr)
print("Período:", train_period.replace("_", "–"))
print("Excel:", xlsx)
print("Rasters de treino:", train_folder)
print("Variáveis LRi:", lr_final_folder)

In [ ]:
#definir os modelos
model_features = {
    "rf_lri_base": [
        "lri_dem.tif",
        "lri_slope.tif",
        "lri_lulc.tif"
    ]
}

In [ ]:
nsets = 10

rows = []

for model_name in model_features:
    for i in range(1, nsets + 1):
        run_name = f"{model_name}_{i}"

        trainref = os.path.join(
            train_folder,
            f"f{train_period}_{i}.tif"
        )

        result_folder = os.path.join(
            rf_folder,
            "class",
            run_name
        )

        os.makedirs(
            result_folder,
            exist_ok=True
        )

        rows.append({
            "status": "run",
            "name": model_name,
            "trainref": trainref,
            "trainfeat": lr_final_folder,
            "predfeat": lr_final_folder,
            "yfile": os.path.join(
                rf_folder,
                "npy",
                f"{run_name}_y.npy"
            ),
            "xfile": os.path.join(
                rf_folder,
                "npy",
                f"{run_name}_x.npy"
            ),
            "model": os.path.join(
                rf_folder,
                "models",
                f"{run_name}.joblib"
            ),
            "ntrees": 200,
            "max_samples": -1,
            "result": os.path.join(
                result_folder,
                f"{run_name}.tif"
            )
        })


df_models = pd.DataFrame(rows)

df_models

In [ ]:
#verficação
missing_train = [
    p for p in df_models["trainref"].tolist()
    if not os.path.exists(p)
]

if missing_train:
    raise FileNotFoundError(
        f"Faltam {len(missing_train)} rasters de treino. Exemplo: {missing_train[0]}"
    )

for model_name, feats in model_features.items():
    for f in feats:
        fpath = os.path.join(lr_final_folder, f)

        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Falta variável LRI: {fpath}")

print("Todos os rasters de treino e variáveis LRI existem.")

In [ ]:
#criar o excel
with pd.ExcelWriter(xlsx) as writer:
    df_models.to_excel(
        writer,
        sheet_name="models",
        index=False
    )

    for model_name, feats in model_features.items():
        df_feats = pd.DataFrame({
            "trainfeat": feats,
            "predfeat": feats
        })

        df_feats.to_excel(
            writer,
            sheet_name=model_name,
            index=False
        )

print("models.xlsx criado:", xlsx)

In [ ]:
import datetime
import rasterio as rio


target_rst = os.path.join(
    rf_folder,
    "target",
    f"rst_ba_{train_period}_target.tif"
)

train_rst = os.path.join(
    train_folder,
    f"f{train_period}_1.tif"
)


rasters = {
    "target": target_rst,
    "primeira réplica": train_rst,
}


for name, path in rasters.items():
    with rio.open(path) as src:
        print(f"\n{name}")
        print("Caminho:", path)
        print("Shape:", src.shape)
        print("Resolução:", src.res)
        print("Bounds:", src.bounds)
        print("CRS:", src.crs)
        print("NoData:", src.nodata)

    print(
        "Modificado:",
        datetime.datetime.fromtimestamp(
            os.path.getmtime(path)
        )
    )